# 01 — MediaPipe Feature Extraction

Ekstraksi 45 fitur pose dari landmark MediaPipe Pose.

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp

# KONFIGURASI PATH DATA DI DRIVE
# 1. Path Input UCF101
PATH_UCF_TRAIN = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/UCF101/train'
PATH_UCF_TEST  = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/UCF101/test'

# 2. Path Input HMDB51 (Video & Split)
PATH_HMDB_VIDEO_ROOT = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/HMDB51/Video'
PATH_HMDB_SPLIT_DIR  = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/HMDB51/Split'

# 3. Path Output (45 Fitur)
PATH_OUTPUT_ROOT = '/content/drive/MyDrive/Skripsi Rizal (3.5 Tahun)/Dataset_NPY_Final_45Fitur'

#INISIALISASI MEDIAPIPE POSE
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, model_complexity=1)

# Menghitung sudut antara tiga titik (a, b, c) dengan b sebagai titik pusat
def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b; bc = c - b
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    angle = np.arccos(cosine_angle)
    return np.degrees(angle)

#Ekstraksi Fitur (36 -> 45)
def process_frame_to_45_features(landmarks):
    # Indeks 12 landmark utama: 11-12(bahu), 13-14(siku), 15-16(pergelangan tangan), 23-24(pinggul), 25-26(lutut), 27-28(pergelangan kaki)
    indices = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
    frame_raw = []

    xs = []
    ys = []

    for idx in indices:
        lm = landmarks[idx]
        frame_raw.extend([lm.x, lm.y, lm.z])
        xs.append(lm.x)
        ys.append(lm.y)

    frame_raw = np.array(frame_raw) # 36 Fitur

    def get_pt(idx): return np.array([frame_raw[idx], frame_raw[idx+1]])

    # Koordinat titik-titik kunci
    l_sh, r_sh = get_pt(0), get_pt(3)
    l_hip, r_hip = get_pt(18), get_pt(21)
    l_knee, r_knee = get_pt(24), get_pt(27)
    l_ank, r_ank = get_pt(30), get_pt(33)

    # Fitur geometris
    ankle_dist = np.abs(frame_raw[30] - frame_raw[33]) # Fitur 37, Jarak Horizontal Pergelangan Kaki
    knee_ver_diff = np.abs(frame_raw[25] - frame_raw[28]) # Fitur 38, Selisih Vertikal Lutut

    # Sudut (Fitur 39-42) - Normalized
    ang_l_knee = calculate_angle(l_hip, l_knee, l_ank)
    ang_r_knee = calculate_angle(r_hip, r_knee, r_ank)
    ang_l_hip = calculate_angle(l_sh, l_hip, l_knee)
    ang_r_hip = calculate_angle(r_sh, r_hip, r_knee)

    # Sudut Batang Tubuh Terhadap Vertikal - Fitur 43
    # Solusi untuk: Situp vs Pushup
    mid_shoulder = (l_sh + r_sh) / 2
    mid_hip = (l_hip + r_hip) / 2
    trunk_vec = mid_shoulder - mid_hip
    vertical_vec = np.array([0, -1])

    dot_product = np.dot(trunk_vec, vertical_vec)
    norm_trunk = np.linalg.norm(trunk_vec)
    if norm_trunk == 0: norm_trunk = 1.0

    # Hitung sudut (0=Tegak, 90=Tidur)
    trunk_angle_rad = np.arccos(np.clip(dot_product / norm_trunk, -1.0, 1.0))
    feat_trunk = np.degrees(trunk_angle_rad) / 180.0 # Normalisasi

    # KNEE SYMMETRY (Simetri Lutut) - Fitur 44
    # Solusi untuk: Squat (Simetris) vs Lunges (Asimetris)
    feat_knee_sym = np.abs(ang_l_knee - ang_r_knee) / 180.0

    # ASPECT RATIO (Rasio Dimensi Tubuh) - Fitur 45
    # Solusi untuk: Berdiri vs Tiduran
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    width = max_x - min_x
    height = max_y - min_y
    if width == 0: width = 0.001

    aspect_ratio = height / width
    # Clip di 3.0 dan normalisasi (biar range 0-1)
    feat_ratio = min(aspect_ratio, 3.0) / 3.0

    # Normalisasi sudut lama
    norm_angles = [ang_l_knee/180.0, ang_r_knee/180.0, ang_l_hip/180.0, ang_r_hip/180.0]

    # GABUNG SEMUA (45 Fitur)
    final_features = np.concatenate([
        frame_raw,           # 0-35 (36 fitur)
        [ankle_dist],        # 36
        [knee_ver_diff],     # 37
        norm_angles,         # 38-41 (4 fitur)
        [feat_trunk],        # 42 (NEW)
        [feat_knee_sym],     # 43 (NEW)
        [feat_ratio]         # 44 (NEW)
    ])

    return final_features

# Proses Satu Video
def process_single_video(video_path, save_path):
    if os.path.exists(save_path): return

    cap = cv2.VideoCapture(video_path)
    video_features = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        if results.pose_landmarks:
            # PANGGIL FUNGSI BARU (45 FITUR)
            feats = process_frame_to_45_features(results.pose_landmarks.landmark)
            video_features.append(feats)
    cap.release()

    if len(video_features) > 0:
        np.save(save_path, np.array(video_features))
        print(f"OK: {os.path.basename(video_path)} ({len(video_features)} frames)")
    else:
        print(f"Gagal/Kosong: {os.path.basename(video_path)}")


#PROSES UCF101
def process_ucf_folder(source_root, split_name):
    print(f"\n MEMPROSES UCF101 - {split_name.upper()}...")
    if not os.path.exists(source_root):
        print(f" Folder UCF {split_name} tidak ditemukan!")
        return

    for class_name in os.listdir(source_root):
        class_path = os.path.join(source_root, class_name)
        if not os.path.isdir(class_path): continue

        target_dir = os.path.join(PATH_OUTPUT_ROOT, split_name, class_name)
        os.makedirs(target_dir, exist_ok=True)

        print(f" Kelas: {class_name}")
        for vid in os.listdir(class_path):
            if vid.endswith(('.avi', '.mp4')):
                src = os.path.join(class_path, vid)
                dst = os.path.join(target_dir, vid.replace('.avi','.npy').replace('.mp4','.npy'))
                process_single_video(src, dst)


#PROSES HMDB51
def process_hmdb_situp():
    print(f"\n MEMPROSES HMDB51 - SITUP...")

    split_file_path = os.path.join(PATH_HMDB_SPLIT_DIR, 'situp_test_split1.txt')
    if not os.path.exists(split_file_path):
        candidates = [f for f in os.listdir(PATH_HMDB_SPLIT_DIR) if 'situp' in f and 'split1' in f]
        if candidates:
            split_file_path = os.path.join(PATH_HMDB_SPLIT_DIR, candidates[0])
        else:
            print(" File Split HMDB Situp tidak ditemukan!")
            return

    print(f"Menggunakan Split File: {os.path.basename(split_file_path)}")

    with open(split_file_path, 'r') as f:
        lines = f.readlines()

    path_situp_video = os.path.join(PATH_HMDB_VIDEO_ROOT, 'situp')
    if not os.path.exists(path_situp_video):
        path_situp_video = PATH_HMDB_VIDEO_ROOT

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 2: continue

        vid_name = parts[0]
        split_id = parts[1]

        split_folder = ""
        if split_id == '1': split_folder = 'train'
        elif split_id == '2': split_folder = 'test'
        else: continue

        src = os.path.join(path_situp_video, vid_name)
        target_dir = os.path.join(PATH_OUTPUT_ROOT, split_folder, 'Situp')
        os.makedirs(target_dir, exist_ok=True)
        dst = os.path.join(target_dir, vid_name.replace('.avi','.npy').replace('.mp4','.npy'))

        if os.path.exists(src):
            process_single_video(src, dst)


#EKSEKUSI
if __name__ == "__main__":
    process_ucf_folder(PATH_UCF_TRAIN, 'train')
    process_ucf_folder(PATH_UCF_TEST, 'test')
    process_hmdb_situp()

    print("\n SELESAI! Dataset NPY Final (45 Fitur) siap di:")
    print(PATH_OUTPUT_ROOT)

## Catatan

Log output ekstraksi dari notebook penelitian tidak disertakan agar repository portfolio tetap ringkas.